<a href="https://colab.research.google.com/github/mimimission/ECON_NLP/blob/main/notebooks/00_setup_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 00 — Setup Colab Environment

## Goal
This notebook prepares the full environment for running the `economic-news-labor-rag` project in Google Colab.

**What this notebook does:**
1. Mounts Google Drive
2. Clones the GitHub repository
3. Installs all required Python packages
4. Creates the full folder structure in Google Drive
5. Checks that required API keys are set
6. Verifies that config files are present

**Run this notebook first, every time you start a new Colab session.**

---

## What can go wrong
- Google Drive not mounted → all subsequent notebooks will fail
- `pip install` fails → dependency conflict; check the error message
- `FRED_API_KEY` missing → Notebooks 01 and 03 will fail
- `DEEPSEEK_API_KEY` missing → Notebook 12 will use template fallback (not fatal)
- Git clone fails → check repo URL and internet connection

---

## What you must inspect
- ✅ Drive mount confirms `/content/drive/MyDrive` is accessible
- ✅ All required packages installed without errors
- ✅ All Drive folders created
- ✅ `FRED_API_KEY` found
- ✅ All 8 config files present

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted at /content/drive


## Step 2 — Clone the repository and install requirements

In [ ]:


# Replace with your GitHub token if you need to authenticate from this notebook.
# Do not commit a real token to the repository.

# Git_key = 'your_github_token_here'



In [ ]:
import os
from google.colab import userdata

# Retrieve secrets from Colab's secret manager and set as environment variables
try:
    os.environ['FRED_API_KEY'] = userdata.get('FRED_API_KEY')
    os.environ['DEEPSEEK_API_KEY'] = userdata.get('DEEPSEEK_API_KEY')

    # Fetch Git_key for repository access
    try:
        Git_key = userdata.get('Git_key')
        print("API keys and Git_key successfully loaded from Secrets.")
    except userdata.SecretNotFoundError:
        print("Warning: GITHUB_TOKEN not found in Secrets.")

except userdata.SecretNotFoundError as e:
    print(f"Warning: {e}. Please ensure the keys are added in the Secrets tab.")

API keys and Git_key successfully loaded from Secrets.


In [ ]:
import os

REPO_DIR = '/content/economic-news-labor-rag'
REPO_URL = f'https://{Git_key}@github.com/mimimission/ECON_NLP.git'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'Pulling updates for {REPO_DIR}...')
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
print(f'Working directory: {os.getcwd()}')

Pulling updates for /content/economic-news-labor-rag...
Already up to date.
/content/economic-news-labor-rag
Working directory: /content/economic-news-labor-rag


In [ ]:
!pip install -r requirements.txt optuna -q
print('All packages installed including optuna.')

All packages installed including optuna.


## Step 3 — Define Drive paths

In [ ]:
import pathlib

DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/labor_news_rag_project')

FOLDERS = [
    DRIVE_ROOT / 'data' / 'raw' / 'fred',
    DRIVE_ROOT / 'data' / 'raw' / 'alfred',
    DRIVE_ROOT / 'data' / 'raw' / 'gdelt',
    DRIVE_ROOT / 'data' / 'interim' / 'macro',
    DRIVE_ROOT / 'data' / 'interim' / 'news',
    DRIVE_ROOT / 'data' / 'features' / 'macro',
    DRIVE_ROOT / 'data' / 'features' / 'news',
    DRIVE_ROOT / 'data' / 'model_ready',
    DRIVE_ROOT / 'artifacts' / 'models',
    DRIVE_ROOT / 'artifacts' / 'evidence_index',
    DRIVE_ROOT / 'outputs' / 'predictions',
    DRIVE_ROOT / 'outputs' / 'metrics',
    DRIVE_ROOT / 'outputs' / 'figures',
    DRIVE_ROOT / 'outputs' / 'tables',
    DRIVE_ROOT / 'outputs' / 'explanations',
    DRIVE_ROOT / 'outputs' / 'logs',
    DRIVE_ROOT / 'outputs' / 'data_quality',
    DRIVE_ROOT / 'approvals',
]

for folder in FOLDERS:
    folder.mkdir(parents=True, exist_ok=True)

print(f'Created {len(FOLDERS)} folders under {DRIVE_ROOT}')

Created 18 folders under /content/drive/MyDrive/labor_news_rag_project


## Step 4 — Verify folder structure

In [ ]:
import pandas as pd

rows = []
for folder in FOLDERS:
    rows.append({'path': str(folder.relative_to(DRIVE_ROOT)), 'exists': folder.exists()})

folder_df = pd.DataFrame(rows)
print(folder_df.to_string(index=False))

missing = folder_df[~folder_df['exists']]
if len(missing) > 0:
    print(f'\nWARNING: {len(missing)} folders were not created!')
    print(missing)
else:
    print('\nAll folders created successfully.')

                    path  exists
           data/raw/fred    True
         data/raw/alfred    True
          data/raw/gdelt    True
      data/interim/macro    True
       data/interim/news    True
     data/features/macro    True
      data/features/news    True
        data/model_ready    True
        artifacts/models    True
artifacts/evidence_index    True
     outputs/predictions    True
         outputs/metrics    True
         outputs/figures    True
          outputs/tables    True
    outputs/explanations    True
            outputs/logs    True
    outputs/data_quality    True
               approvals    True

All folders created successfully.


In [ ]:
import shutil
import pathlib

# Define source and destination
# Based on previous check, the notebooks are in the nested directory
REPO_NOTEBOOKS = pathlib.Path('/content/economic-news-labor-rag/economic-news-labor-rag/notebooks')
if not REPO_NOTEBOOKS.exists():
    REPO_NOTEBOOKS = pathlib.Path('/content/economic-news-labor-rag/notebooks')

DRIVE_NOTEBOOKS = pathlib.Path('/content/drive/MyDrive/labor_news_rag_project/notebooks')
DRIVE_NOTEBOOKS.mkdir(parents=True, exist_ok=True)

print(f"Copying notebooks from {REPO_NOTEBOOKS} to {DRIVE_NOTEBOOKS}...")

count = 0
for nb_path in REPO_NOTEBOOKS.glob('*.ipynb'):
    if nb_path.name != '00_setup_colab.ipynb':
        shutil.copy2(nb_path, DRIVE_NOTEBOOKS / nb_path.name)
        print(f"  Copied: {nb_path.name}")
        count += 1

print(f"\nSuccessfully pulled {count} notebooks to Drive.")

Copying notebooks from /content/economic-news-labor-rag/notebooks to /content/drive/MyDrive/labor_news_rag_project/notebooks...
  Copied: 09_train_xgboost_models.ipynb
  Copied: 02_collect_raw_gdelt.ipynb
  Copied: 11_build_rag_evidence_retrieval.ipynb
  Copied: 07_validate_model_ready_dataset.ipynb
  Copied: 14_create_paper_tables_and_figures.ipynb
  Copied: 10_model_audit_and_error_analysis.ipynb
  Copied: 05_build_news_features.ipynb
  Copied: 03_audit_macro_vintage.ipynb
  Copied: 04_audit_and_clean_news.ipynb
  Copied: 06_build_macro_features_and_target.ipynb
  Copied: 12_generate_deepseek_explanations.ipynb
  Copied: 01_collect_raw_fred_alfred.ipynb
  Copied: 13_evaluate_rag_explanations.ipynb
  Copied: 15_final_reproducibility_check.ipynb
  Copied: 08_train_baseline_models.ipynb

Successfully pulled 15 notebooks to Drive.


In [ ]:
import os

drive_path = '/content/drive/MyDrive/labor_news_rag_project/notebooks'

if os.path.exists(drive_path):
    print(f"Checking files in: {drive_path}")
    files = os.listdir(drive_path)
    if files:
        for f in sorted(files):
            print(f"  [FOUND] {f}")
    else:
        print("The directory exists but it is empty.")
else:
    print(f"The directory {drive_path} does not exist yet.")

Checking files in: /content/drive/MyDrive/labor_news_rag_project/notebooks
  [FOUND] 00_setup_colab.ipynb
  [FOUND] 01_collect_raw_fred_alfred.ipynb
  [FOUND] 02_collect_raw_gdelt.ipynb
  [FOUND] 03_audit_macro_vintage.ipynb
  [FOUND] 04_audit_and_clean_news.ipynb
  [FOUND] 05_build_news_features.ipynb
  [FOUND] 06_build_macro_features_and_target.ipynb
  [FOUND] 07_validate_model_ready_dataset.ipynb
  [FOUND] 08_train_baseline_models.ipynb
  [FOUND] 09_train_xgboost_models.ipynb
  [FOUND] 10_model_audit_and_error_analysis.ipynb
  [FOUND] 11_build_rag_evidence_retrieval.ipynb
  [FOUND] 12_generate_deepseek_explanations.ipynb
  [FOUND] 13_evaluate_rag_explanations.ipynb
  [FOUND] 14_create_paper_tables_and_figures.ipynb
  [FOUND] 15_final_reproducibility_check.ipynb


## Step 5 — Check API keys

In [ ]:
import os

# --- SET YOUR API KEYS HERE ---
# Option A: set them directly (do not commit this cell with keys filled in)
# os.environ['FRED_API_KEY'] = 'your_fred_api_key'
# os.environ['DEEPSEEK_API_KEY'] = 'your_deepseek_api_key'

# Uncomment and fill in your keys locally if needed:
# os.environ['DEEPSEEK_API_KEY'] = 'your_deepseek_api_key'
# os.environ['FRED_API_KEY'] = 'your_fred_api_key'


# Option B: use Colab Secrets (recommended)
# from google.colab import userdata
# os.environ['FRED_API_KEY'] = userdata.get('FRED_API_KEY')
# os.environ['DEEPSEEK_API_KEY'] = userdata.get('DEEPSEEK_API_KEY')

FRED_API_KEY = os.getenv('FRED_API_KEY')
DEEPSEEK_API_KEY = os.getenv('DEEPSEEK_API_KEY')

print('=== API Key Status ===')
if FRED_API_KEY:
    print(f'FRED_API_KEY:     FOUND ({FRED_API_KEY[:6]}...)')
else:
    print('FRED_API_KEY:     MISSING')
    print()
    print('STOP: FRED_API_KEY is required for FRED/ALFRED.')
    print('Get a free key at: https://fred.stlouisfed.org/docs/api/api_key.html')
    print('Set it with: os.environ["FRED_API_KEY"] = "your_key"')

if DEEPSEEK_API_KEY:
    print(f'DEEPSEEK_API_KEY: FOUND ({DEEPSEEK_API_KEY[:6]}...)')
else:
    print('DEEPSEEK_API_KEY: MISSING')
    print('WARNING: DeepSeek explanations will use template fallback (Notebook 12).')
    print('Forecasting and RAG retrieval will still work without this key.')


=== API Key Status ===
FRED_API_KEY:     FOUND (1bb537...)
DEEPSEEK_API_KEY: FOUND (sk-c06...)


## Step 6 — Verify config files

In [ ]:
import pathlib

# Adjusting path to account for nested directory structure observed in ls -R
REPO_DIR = pathlib.Path('/content/economic-news-labor-rag/economic-news-labor-rag')
if not REPO_DIR.exists():
    REPO_DIR = pathlib.Path('/content/economic-news-labor-rag')

CONFIGS_DIR = REPO_DIR / 'configs'

expected_configs = [
    'base.yaml',
    'fred_series.yaml',
    'gdelt_queries.yaml',
    'cleaning_rules_news.yaml',
    'cleaning_rules_macro.yaml',
    'model_config.yaml',
    'evidence_config.yaml',
    'data_contracts.yaml',
]

print(f'=== Config File Check (Source: {CONFIGS_DIR}) ===')
all_ok = True
for cfg in expected_configs:
    path = CONFIGS_DIR / cfg
    status = 'OK' if path.exists() else 'MISSING'
    print(f'  {cfg:35s} {status}')
    if not path.exists():
        all_ok = False

if all_ok:
    print('\nAll config files present.')
else:
    print('\nWARNING: Some config files are missing. Check that the repo was cloned correctly.')

=== Config File Check (Source: /content/economic-news-labor-rag/configs) ===
  base.yaml                           OK
  fred_series.yaml                    OK
  gdelt_queries.yaml                  OK
  cleaning_rules_news.yaml            OK
  cleaning_rules_macro.yaml           OK
  model_config.yaml                   OK
  evidence_config.yaml                OK
  data_contracts.yaml                 OK

All config files present.


## Step 7 — Load and display base config

In [ ]:
import pathlib
import yaml

# Using the logic from Step 6 to find the correct repo root
REPO_DIR = pathlib.Path('/content/economic-news-labor-rag/economic-news-labor-rag')
if not REPO_DIR.exists():
    REPO_DIR = pathlib.Path('/content/economic-news-labor-rag')

base_cfg_path = REPO_DIR / 'configs' / 'base.yaml'

if not base_cfg_path.exists():
    print(f'STOP: base.yaml not found at {base_cfg_path}. Repo structure is unexpected.')
else:
    with open(base_cfg_path) as f:
        base_cfg = yaml.safe_load(f)

    mode = 'TEST (quick run)' if base_cfg.get('test_mode', False) else 'FULL (research run)'
    print(f'=== base.yaml loaded ===')
    print(f'test_mode : {base_cfg.get("test_mode")}  →  {mode}')
    print()
    print('test_sample (active when test_mode: true):')
    for k, v in base_cfg.get('test_sample', {}).items():
        print(f'  {k}: {v}')
    print()
    print('sample (active when test_mode: false):')
    for k, v in base_cfg.get('sample', {}).items():
        print(f'  {k}: {v}')

=== base.yaml loaded ===
test_mode : True  →  TEST (quick run)

test_sample (active when test_mode: true):
  start_date: 2022-01-01
  end_date: 2024-12-31
  train_end: 2022-12-31
  validation_end: 2023-12-31
  test_start: 2024-01-01

sample (active when test_mode: false):
  start_date: 2000-01-01
  end_date: 2024-12-31
  train_end: 2018-12-31
  validation_end: 2021-12-31
  test_start: 2022-01-01


## Step 8 — Verify package imports

In [ ]:
import importlib

packages = [
    'pandas', 'numpy', 'pyarrow', 'requests', 'yaml', 'tqdm',
    'sklearn', 'xgboost', 'statsmodels', 'scipy', 'matplotlib',
    'joblib', 'openai', 'optuna',
]

print('=== Package Import Check ===')
all_ok = True
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, '__version__', 'n/a')
        print(f'  {pkg:15s} {version}')
    except ImportError as e:
        print(f'  {pkg:15s} MISSING ({e})')
        all_ok = False

if all_ok:
    print('\nAll packages available.')
else:
    print('\nWARNING: Some packages failed to import. Re-run: !pip install -r requirements.txt')

=== Package Import Check ===
  pandas          2.2.2
  numpy           2.0.2
  pyarrow         18.1.0
  requests        2.32.4
  yaml            6.0.3
  tqdm            4.67.3
  sklearn         1.6.1
  xgboost         3.3.0
  statsmodels     0.14.6
  scipy           1.16.3
  matplotlib      3.10.0
  joblib          1.5.3
  openai          2.43.0
  optuna          4.9.0

All packages available.


## Setup Summary

Before running Notebook 01, confirm:

| Check | Status |
|-------|--------|
| Google Drive mounted | ✅ / ❌ |
| Repo cloned | ✅ / ❌ |
| Packages installed | ✅ / ❌ |
| Drive folders created | ✅ / ❌ |
| `FRED_API_KEY` set | ✅ / ❌ |
| Config files present | ✅ / ❌ |

If all checks pass, proceed to `01_collect_raw_fred_alfred.ipynb`.